In [7]:
from google.colab import drive
import os

drive.mount('/content/drive')
os.chdir('/content/drive/MyDrive/컴퓨터비전개론')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Import library

In [9]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab.patches import cv2_imshow

# Part1: goodFeaturesToTrack
- Fill the missing part (denoted as ```fill here```) of the code
- We provide procedure comments for complete the function

In [10]:
def goodFeaturesToTrack(image, maxCorners=100, qualityLevel=0.03, blocksize=7):

    # Image bluring wih averaging filter
    # Only cv2.filter2D is allowed for convolution operation!
    if len(image.shape) == 3:
        gray_image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    else:
        gray_image = image
    gray_image = np.float32(gray_image)
    #can i do this?
    averagingfilter=np.ones((blocksize, blocksize), np.float32) / (blocksize*blocksize)
    image = cv2.filter2D(gray_image,-1,averagingfilter)

    # Compute gradients
    sobel_x=np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=np.float32)
    sobel_y=np.array([[-1,-2,-1],[0,0,0],[1,2,1]], dtype=np.float32)
    Ix = cv2.filter2D(image,-1,sobel_x)
    Iy = cv2.filter2D(image,-1,sobel_y)

    # Compute products of gradients at each pixel
    Ixx = Ix*Ix
    Iyy = Iy*Iy
    Ixy = Ix*Iy

    # Compute the sums of products of gradients in local windows
    sumkernel=np.ones((blocksize,blocksize),np.float32)
    Sxx = cv2.filter2D(Ixx,-1,sumkernel)
    Syy = cv2.filter2D(Iyy,-1,sumkernel)
    Sxy = cv2.filter2D(Ixy,-1,sumkernel)

    # Compute the determinant and trace of the matrix M for each pixel
    detM = Sxx*Syy - Sxy**2
    traceM = Sxx + Syy

    # Compute the Harris response with detM and traceM
    alpha=0.04
    harris_response = detM-alpha*traceM**2

    # Threshold the Harris response to find candidate corners
    harris_response[harris_response<0]=0
    max_harris_response=harris_response.max()
    if max_harris_response<=0: #will not be a corner
        return np.empty((0,1,2),dtype=np.float32)
    #complying the return type with others
    threshold=qualityLevel*max_harris_response
    #set threshold as the portion of the max value.
    corners=np.argwhere(harris_response>threshold)
    #shape--> (N,2)
    if corners.size==0: #no corner found
        return np.empty((0,1,2),dtype=np.float32)
    # Sort the corners by Harris response in descending order
    harris_values_at_corners = harris_response[corners[:, 0], corners[:, 1]]
    sorted_indices = np.argsort(harris_values_at_corners)[::-1]
    sorted_corners = corners[sorted_indices]

    # Keep the top 'maxCorners' corners
    selected_corners = sorted_corners[:maxCorners]

    final_corners = np.array(selected_corners)[:, [1, 0]].astype(np.float32) #swap coordinates column,row
    final_corners = final_corners.reshape(-1, 1, 2)

    return final_corners

# Part2: Optical flow with Lukas-Kanade
- Fill the missing part (denoted as ```fill here```) of the code
- We provide procedure comments for complete the function

In [11]:

def optical_flow(old_frame, new_frame, window_size, min_quality):

    feature_list = goodFeaturesToTrack(old_frame, max_corners, min_quality, blocksize)

    w = int(window_size/2)

    # Normalize
    old_frame = old_frame / 255
    new_frame = new_frame / 255

    # Convolve to get gradients w.r.to X, Y and T dimensions
    kernel_x = np.array([[-1, 1],[-1, 1]], dtype=np.float32)*0.25
    kernel_y = np.array([[-1, -1],[1, 1]], dtype=np.float32)*0.25
    kernel_t = np.array([[1, 1],[1, 1]], dtype=np.float32)*0.25

    # cv2.filter2D is allowed for convolution!
    fx = cv2.filter2D(old_frame, -1, kernel_x)+cv2.filter2D(new_frame,-1,kernel_x)
    fy = cv2.filter2D(old_frame, -1, kernel_y)+cv2.filter2D(new_frame,-1,kernel_y)
    ft = cv2.filter2D(new_frame, -1, kernel_t)-cv2.filter2D(old_frame, -1, kernel_t)

    u = np.zeros(old_frame.shape,dtype=np.float32)
    v = np.zeros(old_frame.shape,dtype=np.float32)

    for feature in feature_list:  # for every corner

        i, j = feature.ravel()  # get cordinates of the corners (i,j). i==column, j==row
        i, j = int(i), int(j)  # i,j are floats initially so convert to integer type
        if (i - w < 0 or i + w >= old_frame.shape[1] or
            j - w < 0 or j + w >= old_frame.shape[0]):
            continue
        #exception coding
        I_x = fx[j - w:j + w + 1, i - w:i + w + 1].flatten()
        I_y = fy[j - w:j + w + 1, i - w:i + w + 1].flatten()
        I_t = ft[j - w:j + w + 1, i - w:i + w + 1].flatten()

        b = np.reshape(I_t, (I_t.shape[0], 1))
        A = np.vstack((I_x, I_y)).T
        if A.shape[0] < 2:
            continue
        #exception coding

        U = -np.matmul(np.linalg.pinv(A), b) # Solving for (u,v) i.e., U

        u[j, i] = U[0,0]
        v[j, i] = U[1,0]

    return (u, v)


# Main function
- If part1 and part2 were filled properly, the 'output.avi' will be generated!
- For google colab, as cv2.imshow() is not provided, so please use cv2_imshow (google.colab.patches) instead  

In [14]:
cap = cv2.VideoCapture('/content/drive/MyDrive/컴퓨터비전개론/slow.mp4')


# Check if video opened successfully
if not cap.isOpened():
    print("Error: Could not open video file. Please check the path and file existence.")
    exit()

# Take first frame and find corners in it
ret, old_frame = cap.read()

# Check if frame was read successfully
if not ret:
    print("Error: Could not read the first frame of the video.")
    cap.release()
    exit()

# Width and height of the file to save
width = cap.get(cv2.CAP_PROP_FRAME_WIDTH)
height = cap.get(cv2.CAP_PROP_FRAME_HEIGHT)

# 'output.mp4' will be generated!
fourcc = cv2.VideoWriter_fourcc(*'MP4V')
out = cv2.VideoWriter('output.mp4',  fourcc, 30.0, (int(width), int(height)))

old_gray = cv2.cvtColor(old_frame, cv2.COLOR_BGR2GRAY)

# Shi Tomasi parameter
max_corners = 100
min_quality = 0.3
blocksize = 7
p0 = goodFeaturesToTrack(old_frame, max_corners, min_quality, blocksize)

# Create a mask image for drawing purposes
mask = np.zeros_like(old_frame)

while(1):
    ret, current_frame = cap.read()
    if not ret:
        break
    frame_gray = cv2.cvtColor(current_frame, cv2.COLOR_BGR2GRAY)

    # calculate optical flow
    U, V = optical_flow(old_gray, frame_gray, 15, 0.03)

    for i in range(current_frame.shape[0]):
        for j in range(current_frame.shape[1]):
            u, v = U[i][j], V[i][j]
            if u and v:
                mask = cv2.line(mask, (j, i), (int(round(j + u)), int(round(i + v))), (0, 255, 0), 2)
                frame = cv2.arrowedLine(current_frame, (j, i), (int(round(j + u)), int(round(i + v))), (0, 255, 0), thickness=2)
                current_frame = cv2.add(current_frame, mask)

    # Display the frame with optical flow vectors
    cv2_imshow(current_frame)
    out.write(current_frame)
    # Break the loop if 'Esc' key is pressed
    if cv2.waitKey(30) == 27:
        break

    # Set the current frame as the previous frame for the next iteration
    old_gray = frame_gray

# Release the video capture object
cap.release()
out.release()

# Close the plot window when done
plt.close()
cv2.destroyAllWindows()

Output hidden; open in https://colab.research.google.com to view.